# MultiModal Classification (Text + Image) on Fakeddit Dataset
- Fakeddit is a fine-grained multimodal fake news detection dataset, designed to advance efforts to combat the spread of misinformation in multiple modalities.
- I worked on classifying data into 6 pre-defined classes: authentic/true news content, Satire/Parody, content with false connection, imposter content, manipulated content and misleading content.
- In the Image-Feature Extractor, I used a pre-trained `ResNet50 model` trained on the ImageNet dataset for image classification tasks.
- In the Text-Feature Extractor, I used a pre-trained `Bertmodel` trained on the English Wikipedia and Toronto Book Corpus in lower cased letters.


---


##### Why BERT and BERT embeddings?
*   BERT uses a bi-directional approach considering both the left and right context of words in a sentence, instead of analyzing the text sequentially.
*   We use BERT to extract features, namely word and sentence embedding vectors, from text data.
*   These vectors are used as high-quality feature inputs to downstream models. NLP models such as LSTMs or CNNs require inputs in the form of numerical vectors, hence BERT is a good option for encoding variable length text strings.

---

##### Why use ResNet50?
*   ResNet50 is a deep learning model launched in 2015 by Microsoft Research for the purpose of visual recognition. The model is 50 layers deep.
*   ResNet50's architecture (including shortcut connections between layers) significantly improves on the vanishing gradient problems that arise during backpropagation which allows for higher accuracy.
*   The skip connections in ResNet50 facilitate smoother training and faster convergence. Thus making it easier for the model to learn and update weights during training.


In [ ]:
# Import statements
import os
import random
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from urllib import request
from urllib.error import URLError, HTTPError

import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights, vit_b_16, ViT_B_16_Weights
from transformers import BertTokenizer, BertModel

In [ ]:
# Import Neural Network and PyTorch Libraries

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import torch.optim.lr_scheduler as lr_scheduler
import matplotlib.pyplot as plt

# ---- reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

cudnn.deterministic = True
cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ---- project paths ----
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WORKING_DIR = DATA_DIR / "working"
IMAGE_DIR = WORKING_DIR / "images"
SPLIT_DIR = WORKING_DIR / "splits"
MODEL_DIR = PROJECT_ROOT / "saved_models"
CACHE_DIR = PROJECT_ROOT / "hf_cache"

for p in [DATA_DIR, RAW_DIR, WORKING_DIR, IMAGE_DIR, SPLIT_DIR, MODEL_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---- huggingface cache ----
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR)

# ---- files ----
TSV_PATH = RAW_DIR / "multimodal_train.tsv"   # put your source tsv here
print("TSV_PATH:", TSV_PATH)
print("IMAGE_DIR:", IMAGE_DIR)
print("MODEL_DIR:", MODEL_DIR)

In [ ]:
# ==========================================
# 2. ONE-TIME DATA PREPARATION
# ==========================================

SAMPLE_FRAC = 0.05   # use 0.05 like your previous setup; increase later if compute allows
TEXT_COL = "clean_title"
LABEL_COL = "6_way_label"
ID_COL = "id"
IMAGE_URL_COL = "image_url"

# adjust column names here if your tsv uses the notebook's original names
COLUMN_RENAME = {
    "clean_title": "clean_title",
    "6_way_label": "6_way_label",
    "image_url": "image_url",
    "id": "id",
    "2_way_label": "2_way_label",
    "3_way_label": "3_way_label",
    "title": "title",
    "hasImage": "hasImage"
}

df = pd.read_csv(TSV_PATH, sep="\t")
df = df.rename(columns=COLUMN_RENAME)

# keep only rows with image and required fields
drop_cols = [c for c in ["2_way_label", "3_way_label", "title"] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

required = [TEXT_COL, LABEL_COL, ID_COL, IMAGE_URL_COL]
df = df.dropna(subset=required).copy()

if "hasImage" in df.columns:
    df = df[df["hasImage"] == True].copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""].copy()

# stratified sample
if SAMPLE_FRAC < 1.0:
    df, _ = train_test_split(
        df,
        test_size=1 - SAMPLE_FRAC,
        stratify=df[LABEL_COL],
        random_state=SEED,
        shuffle=True
    )

df = df.reset_index(drop=True)
print("Sampled rows:", len(df))
print(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# ==========================================
# 3. DOWNLOAD IMAGES ONLY IF MISSING
# ==========================================

def download_image_if_needed(img_url: str, img_id: str, image_dir: Path, timeout: int = 20):
    img_path = image_dir / f"{img_id}.jpg"

    if img_path.exists():
        return True, img_path, "cached"

    try:
        with request.urlopen(img_url, timeout=timeout) as response:
            content = response.read()
        with open(img_path, "wb") as f:
            f.write(content)

        # validate immediately
        with Image.open(img_path) as im:
            im.convert("RGB")

        return True, img_path, "downloaded"

    except (HTTPError, URLError, TimeoutError, UnidentifiedImageError, OSError) as e:
        if img_path.exists():
            img_path.unlink(missing_ok=True)
        return False, None, str(e)

valid_rows = []
stats = {"cached": 0, "downloaded": 0, "failed": 0}

for _, row in df.iterrows():
    ok, path, status = download_image_if_needed(
        img_url=row[IMAGE_URL_COL],
        img_id=row[ID_COL],
        image_dir=IMAGE_DIR
    )
    if ok:
        valid_rows.append(row)
        stats[status] += 1
    else:
        stats["failed"] += 1

df = pd.DataFrame(valid_rows).reset_index(drop=True)
print(stats)
print("Rows after valid-image filtering:", len(df))


In [ ]:
# ==========================================
# 4. FINAL IMAGE VALIDATION
# ==========================================

good_idx = []

for i, row in df.iterrows():
    img_path = IMAGE_DIR / f"{row[ID_COL]}.jpg"
    try:
        with Image.open(img_path) as im:
            im.verify()
        good_idx.append(i)
    except Exception:
        pass

df = df.iloc[good_idx].reset_index(drop=True)
print("Rows after final validation:", len(df))
print(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# ==========================================
# 5. SPLITS + SAVE CLEAN CSVs
# ==========================================

df_train, df_temp = train_test_split(
    df,
    test_size=0.20,
    stratify=df[LABEL_COL],
    random_state=SEED
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    stratify=df_temp[LABEL_COL],
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df.to_csv(SPLIT_DIR / "full_clean_sample.csv", index=False)
df_train.to_csv(SPLIT_DIR / "train.csv", index=False)
df_val.to_csv(SPLIT_DIR / "val.csv", index=False)
df_test.to_csv(SPLIT_DIR / "test.csv", index=False)

print(len(df_train), len(df_val), len(df_test))


In [ ]:
# ==========================================
# 6. TOKENIZER + DATASET
# ==========================================

MAX_LEN = 80
BATCH_SIZE = 16
NUM_CLASSES = 6

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", cache_dir=str(CACHE_DIR))

class FakedditDataset(Dataset):
    def __init__(self, df, image_dir, tokenizer, max_len=80):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row[TEXT_COL])
        label = int(row[LABEL_COL])
        img_path = self.image_dir / f"{row[ID_COL]}.jpg"

        if not img_path.exists():
            raise FileNotFoundError(f"Missing image: {img_path}")

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Bad image file: {img_path} | {e}")

        image = self.transform(image)

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "image": image,
            "label": torch.tensor(label, dtype=torch.long)
        }


df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_val = pd.read_csv(SPLIT_DIR / "val.csv")
df_test = pd.read_csv(SPLIT_DIR / "test.csv")

train_dataset = FakedditDataset(df_train, IMAGE_DIR, tokenizer, MAX_LEN)
val_dataset = FakedditDataset(df_val, IMAGE_DIR, tokenizer, MAX_LEN)
test_dataset = FakedditDataset(df_test, IMAGE_DIR, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["image"].shape, batch["label"].shape)


In [ ]:
sample = train_dataset[0]
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["image"].shape)
print(sample["label"])


In [ ]:
print(len(train_dataset))
print(train_dataset[0]["image"].shape)
print((IMAGE_DIR / f"{df_train.iloc[0][ID_COL]}.jpg").exists())


In [ ]:
# ==========================================
# 7. TEXT-ONLY BERT BASELINE
# ==========================================

from transformers import BertModel
import torch
import torch.nn as nn

class TextOnlyBERTClassifier(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        x = self.dropout(pooled)
        logits = self.classifier(x)
        return logits


In [ ]:
# ==========================================
# 8. IMAGE-ONLY RESNET50 BASELINE
# ==========================================

class ImageOnlyResNetClassifier(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3, freeze_backbone=False):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2
        self.backbone = resnet50(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, image):
        feats = self.backbone(image)
        feats = self.dropout(feats)
        logits = self.classifier(feats)
        return logits


In [ ]:
# ==========================================
# 9. BERT + RESNET FUSION MODEL
# ==========================================

class BERTResNetFusionClassifier(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3, freeze_bert=False, freeze_resnet=False):
        super().__init__()

        # text branch
        self.bert = BertModel.from_pretrained("bert-base-uncased", cache_dir=str(CACHE_DIR))
        bert_dim = self.bert.config.hidden_size

        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        # image branch
        weights = ResNet50_Weights.IMAGENET1K_V2
        self.resnet = resnet50(weights=weights)
        resnet_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        if freeze_resnet:
            for p in self.resnet.parameters():
                p.requires_grad = False

        self.text_proj = nn.Linear(bert_dim, 256)
        self.image_proj = nn.Linear(resnet_dim, 256)

        self.dropout = nn.Dropout(dropout)

        self.fusion = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        text_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.pooler_output
        text_feat = self.dropout(self.text_proj(text_feat))

        image_feat = self.resnet(image)
        image_feat = self.dropout(self.image_proj(image_feat))

        fused = torch.cat([text_feat, image_feat], dim=1)
        logits = self.fusion(fused)

        return logits


In [ ]:
# ==========================================
# 10. EARLY STOPPING
# ==========================================

class EarlyStopping:
    def __init__(self, patience=4, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def step(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            return False

        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

        return self.should_stop


In [ ]:
# ==========================================
# 11. METRICS / TRAIN / EVAL HELPERS
# ==========================================

def get_class_weights(df, label_col):
    classes = np.sort(df[label_col].unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=df[label_col].values
    )
    return torch.tensor(weights, dtype=torch.float32).to(device)

def move_batch_to_device(batch):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "accuracy": acc,
        "precision_weighted": p_w,
        "recall_weighted": r_w,
        "f1_weighted": f1_w,
        "precision_macro": p_m,
        "recall_macro": r_m,
        "f1_macro": f1_m
    }


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, checkpoint_name):
    early_stopper = EarlyStopping(patience=4, min_delta=0.0)
    best_val_loss = float("inf")
    history = []

    checkpoint_path = MODEL_DIR / checkpoint_name
    saved_any_checkpoint = False

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_true, train_pred = [], []

        for batch in train_loader:
            batch = move_batch_to_device(batch)

            optimizer.zero_grad()

            if isinstance(model, TextOnlyBERTClassifier):
                logits = model(batch["input_ids"], batch["attention_mask"])
            elif isinstance(model, ImageOnlyResNetClassifier):
                logits = model(batch["image"])
            else:
                logits = model(batch["image"], batch["input_ids"], batch["attention_mask"])

            if not torch.isfinite(logits).all():
                raise ValueError("Non-finite logits detected during training.")

            loss = criterion(logits, batch["label"])

            if not torch.isfinite(loss):
                raise ValueError("Non-finite loss detected during training.")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * batch["label"].size(0)

            preds = torch.argmax(logits, dim=1)
            train_true.extend(batch["label"].detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_metrics = compute_metrics(train_true, train_pred)

        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []

        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch_to_device(batch)

                if isinstance(model, TextOnlyBERTClassifier):
                    logits = model(batch["input_ids"], batch["attention_mask"])
                elif isinstance(model, ImageOnlyResNetClassifier):
                    logits = model(batch["image"])
                else:
                    logits = model(batch["image"], batch["input_ids"], batch["attention_mask"])

                if not torch.isfinite(logits).all():
                    raise ValueError("Non-finite logits detected during validation.")

                loss = criterion(logits, batch["label"])

                if not torch.isfinite(loss):
                    raise ValueError("Non-finite val loss detected during validation.")

                val_loss += loss.item() * batch["label"].size(0)

                preds = torch.argmax(logits, dim=1)
                val_true.extend(batch["label"].detach().cpu().numpy())
                val_pred.extend(preds.detach().cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_metrics = compute_metrics(val_true, val_pred)
        scheduler.step(val_loss)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()}
        })

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | val_f1_macro={val_metrics['f1_macro']:.4f}"
        )

        if torch.isfinite(torch.tensor(val_loss)) and val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            saved_any_checkpoint = True
            print(f"Saved best model to: {checkpoint_path}")

        if early_stopper.step(val_loss):
            print("Early stopping triggered.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(MODEL_DIR / f"{checkpoint_name.replace('.pth', '')}_history.csv", index=False)

    if not saved_any_checkpoint:
        fallback_path = MODEL_DIR / f"last_{checkpoint_name}"
        torch.save(model.state_dict(), fallback_path)
        print(f"No best checkpoint was saved; wrote fallback checkpoint to: {fallback_path}")

    return history_df


In [ ]:
# ==========================================
# 13. TEST EVALUATION
# ==========================================

def evaluate_model(model, test_loader, checkpoint_name):
    checkpoint_path = MODEL_DIR / checkpoint_name
    fallback_path = MODEL_DIR / f"last_{checkpoint_name}"

    if checkpoint_path.exists():
        load_path = checkpoint_path
    elif fallback_path.exists():
        load_path = fallback_path
    else:
        raise FileNotFoundError(
            f"No checkpoint found. Checked:\n{checkpoint_path}\n{fallback_path}"
        )

    model.load_state_dict(torch.load(load_path, map_location=device))
    model.eval()

    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = move_batch_to_device(batch)

            if isinstance(model, TextOnlyBERTClassifier):
                logits = model(batch["input_ids"], batch["attention_mask"])
            elif isinstance(model, ImageOnlyResNetClassifier):
                logits = model(batch["image"])
            else:
                logits = model(batch["image"], batch["input_ids"], batch["attention_mask"])

            preds = torch.argmax(logits, dim=1)
            y_true.extend(batch["label"].cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    metrics = compute_metrics(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print("Test metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    print("\nConfusion matrix:\n", cm)

    return metrics, cm


In [ ]:
# ==========================================
# 14. SHARED TRAINING SETTINGS
# ==========================================

class_weights = get_class_weights(df_train, LABEL_COL)
criterion = nn.CrossEntropyLoss(weight=class_weights)
EPOCHS = 10


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.cuda.empty_cache()

# Make sure class weights are safe
class_weights = class_weights.to(device=device, dtype=torch.float32)

# Fresh frozen-BERT text model
text_model = TextOnlyBERTClassifier(
    num_classes=NUM_CLASSES,
    dropout=0.3,
    freeze_bert=True
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, text_model.parameters()),
    lr=1e-4,
    eps=1e-8,
    weight_decay=0.0,
    foreach=False,
    fused=False
)

print("device:", device)
print("class_weights dtype:", class_weights.dtype)

# Check parameter dtypes
for n, p in text_model.named_parameters():
    if p.requires_grad:
        print("trainable:", n, p.dtype, p.shape)
print("-" * 60)

text_model.train()

for step, batch in enumerate(train_loader):
    if step >= 20:
        break

    batch = move_batch_to_device(batch)

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    labels = batch["label"].long()

    if step == 0:
        print("input_ids dtype:", input_ids.dtype)
        print("attention_mask dtype:", attention_mask.dtype)
        print("labels dtype:", labels.dtype)
        print("labels min/max:", labels.min().item(), labels.max().item())

    optimizer.zero_grad(set_to_none=True)

    logits = text_model(input_ids, attention_mask)

    if not torch.isfinite(logits).all():
        raise ValueError(f"Non-finite logits at step {step}")

    loss = criterion(logits, labels)

    if not torch.isfinite(loss):
        raise ValueError(f"Non-finite loss at step {step}: {loss.item()}")

    loss.backward()

    bad_grad_name = None
    for n, p in text_model.named_parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            bad_grad_name = n
            break
    if bad_grad_name is not None:
        raise ValueError(f"Non-finite gradient at step {step}: {bad_grad_name}")

    grad_norm = torch.nn.utils.clip_grad_norm_(text_model.parameters(), max_norm=1.0)

    optimizer.step()

    bad_param_name = None
    for n, p in text_model.named_parameters():
        if not torch.isfinite(p).all():
            bad_param_name = n
            break
    if bad_param_name is not None:
        raise ValueError(f"Non-finite parameter after step {step}: {bad_param_name}")

    preds = torch.argmax(logits, dim=1)
    acc = (preds == labels).float().mean().item()

    print(
        f"step={step:02d} | "
        f"loss={loss.item():.4f} | "
        f"acc={acc:.4f} | "
        f"grad_norm={float(grad_norm):.4f}"
    )

print("PASSED: 20 local training steps with frozen BERT.")


In [ ]:
text_model = TextOnlyBERTClassifier(num_classes=NUM_CLASSES, dropout=0.3).to(device)

optimizer = optim.Adam(text_model.parameters(), lr=2e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

text_history = train_model(
    model=text_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    checkpoint_name="text_only_bert.pth"
)

text_metrics, text_cm = evaluate_model(text_model, test_loader, "text_only_bert.pth")


In [ ]:
image_model = ImageOnlyResNetClassifier(
    num_classes=NUM_CLASSES,
    dropout=0.3,
    freeze_backbone=False
).to(device)

optimizer = optim.Adam(image_model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

image_history = train_model(
    model=image_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    checkpoint_name="image_only_resnet50.pth"
)

image_metrics, image_cm = evaluate_model(image_model, test_loader, "image_only_resnet50.pth")


In [ ]:
fusion_model = BERTResNetFusionClassifier(
    num_classes=NUM_CLASSES,
    dropout=0.3,
    freeze_bert=False,
    freeze_resnet=False
).to(device)

optimizer = optim.Adam(fusion_model.parameters(), lr=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

fusion_history = train_model(
    model=fusion_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    checkpoint_name="bert_resnet_fusion.pth"
)

fusion_metrics, fusion_cm = evaluate_model(fusion_model, test_loader, "bert_resnet_fusion.pth")


Optional VIT upgrade

In [ ]:
# ==========================================
# 15. OPTIONAL: IMAGE-ONLY VIT
# ==========================================

class ImageOnlyViTClassifier(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3, freeze_backbone=False):
        super().__init__()
        weights = ViT_B_16_Weights.IMAGENET1K_V1
        self.backbone = vit_b_16(weights=weights)
        in_features = self.backbone.heads.head.in_features
        self.backbone.heads = nn.Identity()

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, image):
        feats = self.backbone(image)
        feats = self.dropout(feats)
        logits = self.classifier(feats)
        return logits


In [ ]:
# ==========================================
# 16. OPTIONAL: BERT + VIT FUSION
# ==========================================

class BERTViTFusionClassifier(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3, freeze_bert=False, freeze_vit=False):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased", cache_dir=str(CACHE_DIR))
        bert_dim = self.bert.config.hidden_size
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        weights = ViT_B_16_Weights.IMAGENET1K_V1
        self.vit = vit_b_16(weights=weights)
        vit_dim = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()

        if freeze_vit:
            for p in self.vit.parameters():
                p.requires_grad = False

        self.text_proj = nn.Linear(bert_dim, 256)
        self.image_proj = nn.Linear(vit_dim, 256)
        self.dropout = nn.Dropout(dropout)

        self.fusion = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        text_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = self.dropout(self.text_proj(text_outputs.pooler_output))

        image_feat = self.dropout(self.image_proj(self.vit(image)))

        fused = torch.cat([text_feat, image_feat], dim=1)
        logits = self.fusion(fused)
        return logits


#### Verifying that code is running on TPU
- Running on CPU was very slow so I switched to TPU as my hardware accelerator on Colab (whenever available) to speed up training
- My running speed reduced by upto 3x upon switching to TPU

Why TPU vs GPU or CPU?
- TPU is designed for tensor processing and neural network loads
- They are highly efficient for deep learning tasks that involve matrix operations.
- Since my model operates primarily on CNNs and tensors, I found it to significantly optimize running time

If not TPU, using GPU T4x2 for training

In [ ]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("CUDA is available. Using GPU.")
else:
    device = torch.device('cpu')
    print("CUDA is not available. Using CPU.")

#### Importing Data

In [ ]:
df = pd.read_csv('F:/M.Sc. Electronics Information Engineering/Advanced Ai/Project/fake-news-meme-detector/data/multimodal_only_samples/multimodal_train.tsv', sep='\t')

**Cleaning up the dataframe:**
1. Since I am working on 6-way classification, removing the columns for 2-way and 3-way classification
2. There is a cleaned title column available with filtered text so we drop the unused title column
  - The `'Bert-base-uncased'` tokenizer that I use for text encoding and classification is specifically designed for handling lowercase text, hence the requirement for clean title column


In [ ]:
df.drop(['2_way_label', '3_way_label', 'title'], axis = 1, inplace =True)

In [ ]:
df.head(10)

- Applying `train_test_split()` function in order to minimize complete dataset with over 700.000 samples.

- I split the complete dataset into custom training and test sets, while reusing only 10% of the original data.
- Stratify function is applied in order to keep the per class sample distribution from original Fakeddit source dataset.

In [ ]:

df, df_backup = train_test_split(
    df,
    test_size=0.95,
    shuffle=True,
    # To maintain percentage of samples per class as given by original dataset
    stratify=df["6_way_label"]
)

#### Data visualization

In [ ]:
# Reset indexes as we are now working with a smaller sample of original dataset
df.reset_index(drop=True, inplace=True)
df

Checking for NaN values in relevant columns of DataFrame

In [ ]:
print(df['clean_title'].isnull().sum())
print(df['id'].isnull().sum())
print(df['hasImage'].isnull().sum())

# Check for how many rows the column hasImage would be False
print(df['hasImage'].value_counts())

**Plotting the 6- Way Class Distribution of dataset:**
- 0: TRUE
- 1: SATIRE
- 2: FALSE CONNECTION
- 3: IMPOSTER CONTENT
- 4: MANIPULATED CONTENT
- 5: MISLEADING CONTENT

In [ ]:
from matplotlib import pyplot as plt
df['6_way_label'].plot(kind='hist', bins=20, title='6_way_label')

> Based on the class distribution graph, we can say that the classes are not represented equally in the dataset.

> Due to this, I applied balanced class weights later on in my loss function

#### Importing images from url into working directory

- Used the `request` module from `urllib` library to handle the downloading of images from URLs.
- Any NaN values in the DataFrame are replaced with empty strings using `replace()` and `fillna()` methods. This ensures that there are no missing values when working with the data.
- Used `urlopen()` to download the image from the URL and write the content to a file. The file is saved with the id of the row and the extension .jpg, to enable easy future access
- I found that some image URLs do not exist anymore/ have been taken down, the corresponding rows are dropped from the DataFrame.

In [ ]:
from urllib import request

# Replace NaN values with empty strings
df = df.replace(np.nan, '', regex=True)
df.fillna('', inplace=True)

# Make a directory to download images into
if not os.path.exists("data/working/images"):
  os.makedirs("data/working/images")

for index, row in df.iterrows():
  if row["hasImage"] == True and row["image_url"] != "" and row["image_url"] != "nan":
    image_url = row["image_url"]
    path = "data/working/images/" + row["id"] + ".jpg"

    try:
      f = open(path, 'wb')
      f.write(request.urlopen(image_url).read())
      f.close()

    except:
        # To account for now invalid image urls
        df.drop(index = index, axis = 0, inplace = True)
        pass

print("Downloaded all images.")
df.reset_index(drop=True, inplace=True)

In [ ]:
# Plotting images to test download
for i in range(5):
    path = "data/working/images/" + df["id"][i] + ".jpg"

    im= np.array(Image.open(path))

    print(im.shape)
    ax= plt.subplot(121)
    ax.imshow(im)

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load the RGBA image
image_path = "data/working/images/" + df["id"][0] + ".jpg"
image = Image.open(image_path).convert("RGB")

# Split the image into individual channels
r, g, b = image.split()

# Plot each channel separately
plt.figure(figsize=(10, 5))

plt.subplot(1, 4, 1)
plt.imshow(r)
plt.title('Red Channel')

plt.subplot(1, 4, 2)
plt.imshow(g)
plt.title('Green Channel')

plt.subplot(1, 4, 3)
plt.imshow(b)
plt.title('Blue Channel')

#plt.subplot(1, 4, 4)
#plt.imshow(a)
#plt.title('Alpha Channel')

plt.tight_layout()
plt.show()

#### We have images of different sizes which will not work for a CNN


*   Checking for corrupt image files to avoid errors
*   Resizing images in (256, 256, 3) format
*   Testing images





> While attempting to resize images, I was facing errors with some corrupt image files (presumably downloaded from now-defunct links) so I run through image dataset to ensure all images can be successfully opened, else I dropped the corresponding rows from the Dataframe.



In [ ]:
def validate_images(directory):
    corrupted_files = []

    # Walk through directory and sub-directories
    for index, row in df.iterrows():
      image_path = "data/working/images/" + row["id"] + ".jpg"
      try:
          with Image.open(image_path) as img:
              img.verify()
      except Exception as e:
          corrupted_files.append(image_path)
          print(f"Error with {image_path}: {e}")
          df.drop(index = index, axis = 0, inplace = True)

    return corrupted_files

# Example usage:
directory = "data/working/images/"
corrupted_images = validate_images(directory)
if corrupted_images:
    print(f"Found {len(corrupted_images)} corrupted images.")
else:
    print("All images are valid!")
df.reset_index(drop=True, inplace=True)

Resized all the images in dataset to a standard size of (256, 256) using PyTorch's `torchvision.transforms`. The resized images overwrite the original ones.
- Each image is converted to RGB format to ensure consistent color channels.
- Used PyTorch's `torchvision.transforms.Resize` to resize the image to the specified new_size.
- This is so I can pass all the images as uniform (256, 256, 3) matrixes in the CNN.

In [ ]:
# Resizing all the images to a standard (256,256, 3) using pytorch

# Define the desired size
new_size = (256, 256)

for index, row in df.iterrows():
    image_path = "data/working/images/" + row["id"] + ".jpg"
    image = Image.open(image_path).convert("RGB")

    # Resize the image using PyTorch's torchvision.transforms
    resize_transform = v2.Resize(new_size)
    resized_image = resize_transform(image)
    resized_image.save(image_path)

In [ ]:
# Plotting images to test resize
for i in range(5):
    path = "data/working/images/" + df["id"][i] + ".jpg"

    im= np.array(Image.open(path))

    print(im.shape)
    ax= plt.subplot(121)
    ax.imshow(im)

    plt.show()

#### Defining BERT Tokenizer to convert variable length clean-title strings to uniform 768-element arrays

- Using a pre-trained `BERT model` from the Hugging Face `transformers` library, I generated BERT embeddings for the Reddit title strings.
- BERT BASE has a feed-forward network of 768 hidden layers which results in a uniform length array everytime.
- These arrays can then be used in a Neural Network for classification tasks

In [ ]:
%%capture


! pip install bert-serving-server  # server-side
! pip install bert-serving-client  # client-side
! pip install torch transformers

In [ ]:
import torch
from transformers import BertModel, BertTokenizer

# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name, output_hidden_states = True)

# Put the model in evaluation mode, which turns off dropout regularization which is used in training.
bert_model.eval()

In [ ]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        return_tensors="pt",
        max_length=80,
        truncation=True,
        padding="max_length",
        return_attention_mask=True
    )

    return inputs["input_ids"].squeeze(0), inputs["attention_mask"].squeeze(0)

# test
text = "This is an example Reddit submission title."
input_ids, attention_mask = get_bert_embedding(text)
print(input_ids.shape)        # torch.Size([80])
print(attention_mask.shape)   # torch.Size([80])


#### Loading and Processing Input Data

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df["6_way_label"])
df_test, df_val = train_test_split(df_test, test_size=0.5, stratify=df_test["6_way_label"])

> Created `FakedditDataset` class, inherited from PyTorch's `Dataset` class
  - Applies transforms to image
  - Returns:
    - `Image` in form of tensor
    - BERT tokenizer `input_ids` and `attention_masks`
    - corresponding 6-way `label`

> Upon applying normalization, model returned poor results (predicted all 0's). Hence, it's one of the transforms I decided not to apply

> FakedditDataset contains all relevant information per batch and DataLoader iterates over complete Dataset to feed batches of *size 16* to model

In [ ]:
class FakedditDataset(Dataset):
    def __init__(self, df, text_field="clean_title", label_field="6_way_label", image_id="id"):
        self.df = df.reset_index(drop=True)
        self.text_field = text_field
        self.label_field = label_field
        self.image_id = image_id

        self.img_size = 256
        # Using the pre-calculated ImageNet mean and std values for normalization
        self.mean, self.std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        self.transform_func = v2.Compose(
                [   v2.Resize(256),
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),
                    v2.Normalize(self.mean, self.std)
                    ])

    def __getitem__(self, index):
        text = str(self.df.at[index, self.text_field])
        label = self.df.at[index, self.label_field]
        img_path = "data/working/images/" + self.df.at[index, self.image_id] + ".jpg"

        image = Image.open(img_path)
        img = self.transform_func(image)

        input_ids, attention_mask = get_bert_embedding(text)

        return input_ids, attention_mask, label, img

    def __len__(self):
        return self.df.shape[0]

In [ ]:
train_dataset = FakedditDataset(df_train)
test_dataset = FakedditDataset(df_test)
val_dataset = FakedditDataset(df_val)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)

print(len(train_loader))

# Verifying dataset was created accurately
input_ids, attention_mask, label, img = next(iter(train_loader))
print(input_ids.shape, attention_mask.shape, label.shape, img.shape)

#### Building the Model #1

`MultiModalClassifier` class processes post title and image data using `Bert-Base model` for processing text data and a `simple CNN-based model` for processing image data, and is derived of `nn.Module` model super class from torch.nn module.

##### Convolutional Neural Network:
*   The architecture of my model is derived from standard CNNs built to perform image classification tasks
*   I designed a simple architecture consisting of three convolutional layers to perform feature extraction with the following composition:
  - **2D convolution**: with a (3x3) kernel and 1 unit of padding.
  - **Activation**: ReLU is applied to introduce non-linearity.
  - **MaxPooling**: This layer reduces the spatial dimensions by half while maintaining number of channels.
*   Fully Connected Layer:
  - **Input**: The output from the last convolutional layer is flattened into a vector of size 128 * 32 * 32 = 131072.
  - **Output**: This vector is passed through a fully connected layer (linear layer) that maps it to a vector of size equal to the number of classes (num_classes).
  - **Activation**: ReLU is applied to the output of this layer.


---

##### Text Processing Branch
- To process the text, I utilized a pre-trained BERT model (`bert-base-uncased`) to process the text input.
- The BertModel outputs a 768-dimensional vector for each input token

- Linear Layer: Applied a fully connected layer to reduce the dimensionality of the BERT output to match the number of target classes (= 6).

---

*  I implemented a `dropout()` layer to prevent overfitting
*  I used the `torch.max()` function to merge the features from the text and image branches of multimodal model. This approach selects the maximum value between the corresponding elements of `x_text` and `x_img`.
  - This yielded better results than `torch.cat()` and `torch.mean()` to combine the layers of text and image
*   **Activation Function**: softmax()
  - Applied alongside CrossEntropyLoss loss function




In [ ]:
class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=6):

        super(MultimodalClassifier, self).__init__()

        self.num_classes = num_classes

        # Image processing (Simple CNN)
        self.image_conv = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1), # output-> (32, 256, 256)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # output-> (32, 128, 128)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),  # output-> (64, 128, 128)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # output-> (64, 64, 64)
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),  # output-> (128, 64, 64)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # output-> (128, 32, 32)
        )

        # Image processing (Fully Connected Layers)
        self.image_fc = nn.Sequential(
            nn.Linear(128 * 32 * 32, num_classes),  # Assuming input images are 256x256
            nn.ReLU()
        )

        # Dropout layer
        self.drop = nn.Dropout(p=0.3)

        # Text processing branch (using the 768-dimensional BERT arrays)
        self.text_model = BertModel.from_pretrained("bert-base-uncased")
        self.fc_text = nn.Linear(in_features=self.text_model.config.hidden_size, out_features=num_classes, bias=True)

        # Fusion and classification
        self.softmax = nn.Softmax(dim=1)

    def forward(self, image, text_input_ids, text_attention_mask,):
        # Image branch
        x_img = self.image_conv(image)
        x_img = x_img.view(x_img.size(0), -1)  # Flatten the feature maps
        x_img = self.image_fc(x_img)
        x_img = self.drop(x_img)

        # Text branch
        x_text_last_hidden_states = self.text_model(
            input_ids = text_input_ids,
            attention_mask = text_attention_mask,
            return_dict=False
        )
        x_text_pooled_output = x_text_last_hidden_states[0][:, 0, :]
        x_text = self.fc_text(x_text_pooled_output)
        x_text = self.drop(x_text)

        # Fusion and max merge
        x = torch.max(x_text, x_img)

        # Classification
        #x = self.softmax(x) #-> already applied in crossentropy loss

        return x

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MultimodalClassifier(num_classes=6)
model= model.to(device)

#### Training and Evaluating the Model

Training Loop for Model:

*   **Loss Function**: `Weighted Cross Entropy Loss`, it also applies softmax activation function to return final probability distribution of classes
  *   I applied the `get_class_weights()` function to calculate percentage values per class for a weighted CrossEntropy. Thus is because of the highly imbalanced Fakeddit dataset. Since, some classes have considerably more sample data, all classes are weighted and taken as input into the loss calculation according to their respective number of samples.
*   **Early Stopping**: To stop training if loss is increasing
*   **Optimizer**: Adam
*   **Learning Rate**: 1e-4
*   **Epochs:** 5




In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
labels = df_train['6_way_label'].to_numpy()
type(labels)

In [ ]:
# Define loss function and optimizer
from sklearn.utils.class_weight import compute_class_weight

# Assuming 'labels' is a list of all labels in the dataset
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Define the loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, min_lr=1e-6, factor=0.5, patience=1, verbose=True)
num_epochs = 20

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    early_stopping = EarlyStopping(patience=5, verbose=True)
    
  # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for input_ids, attention_mask, label, img in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            label = label.to(device)
            img = img.to(device)
                
            optimizer.zero_grad()

          # Forward pass
            outputs = model(img, input_ids, attention_mask)
            loss = criterion(outputs, label)

          # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()* img.size(0)
            
       # Validating model and ensuring loss is decreasing     
        model.eval()
        val_loss = 0.0
        correct_preds = 0
        with torch.no_grad():
            for input_ids, attention_mask, label, img in val_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                label = label.to(device)
                img = img.to(device)
    
                outputs = model(img, input_ids, attention_mask)
                loss = criterion(outputs, label)
                val_loss += loss.item() * img.size(0)

                _, preds = torch.max(outputs, 1)
                correct_preds += torch.sum(preds == label)

        val_loss = val_loss / len(val_loader.dataset)
        accuracy = correct_preds.double() / len(val_loader.dataset)
        scheduler.step(val_loss)
        print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_loader.dataset):.4f}, Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}')

        # Early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

    torch.save(model.state_dict(), 'Fake-News-Multimodal-Classification/Fakeddit-WebApp/m1.pth')

Evaluation loop for model:

In [ ]:
from sklearn.metrics import precision_score, recall_score

def evaluate_model(model, test_loader, criterion):
    model.eval()
    val_losses = []
    correct_preds = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, attention_mask, label, img in test_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            label = label.to(device)
            img = img.to(device)

            outputs = model(
                  image = img,
                  text_input_ids = input_ids,
                  text_attention_mask = attention_mask
            )

            # Final Softmax layer returns class predictions per sample in batch
            # Highest probability value resembles class prediction and is assigned to preds variable
            _, preds = torch.max(outputs, dim=1)
            print(outputs)

            # Loss is calculated by applying Cross Entropy Loss
            val_loss = criterion(outputs, label)

            # Counting correct model predictions and incrementing correct prediction count
            correct_preds += torch.sum(preds == label)
            print(preds, label)

            # Appending current loss per batch to list of losses per epoch
            val_losses.append(val_loss.item())
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            

    accuracy = float((correct_preds.double() / len(df_test)) * 100)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    print("\nAccuracy: ", accuracy)
    print("Precision: ", precision)
    print("Recall: ", recall)

In [ ]:
#train_model(model, train_loader, criterion, optimizer, num_epochs)
print("\n")
#evaluate_model(model, test_loader, criterion)

Post training insights for BERT + CNN multimodal model:
- NOTE: ran only 4 epochs to get faster results for these insights
- Ran on approx. 2000 rows of data

```
Data:

# During training:
Epoch [1/4], Loss: 1.7745
Epoch [2/4], Loss: 1.7183
Epoch [3/4], Loss: 1.5023
Epoch [4/4], Loss: 1.2373

# Accuracy:
Accuracy:  58.994708994709

#Sample test output:
-> model output: tensor([1, 0, 0, 0, 1, 0, 0, 0, 2, 1, 2, 1, 2, 2, 0, 2])
-> expected output: tensor([1, 5, 0, 4, 2, 3, 0, 0, 2, 0, 2, 1, 0, 2, 2, 0])
```



##### Why switch to ResNets vs shallow CNNs?
*   Residual Network (ResNet) are a Convolutional Neural Network (CNN) architecture that overcomes the “vanishing gradient” problem using shortcut connections
*   ResNets have shown better accuracy than shallow CNNs when training on smaller image datasets.
*   Compared to other CNN models, ResNet-50 has a smaller loss value and converges faster.
*   ResNet50 achieves better parameter efficiency due to its bottleneck architecture, where 1x1 convolutions are used to reduce and then restore the dimensionality of the data

---

*  I continued to use the `torch.max()` function to merge the features from the text and image branches of multimodal model.
*   **Activation Function**: `softmax()`
  - Applied alongside CrossEntropyLoss loss function




In [ ]:
class BERTResNetClassifier(nn.Module):
    def __init__(self, num_classes=6):

        super(BERTResNetClassifier, self).__init__()

        self.num_classes = num_classes

        # Image processing (ResNet)
        self.image_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        # Image processing (Fully Connected Layer)
        self.fc_image = nn.Linear(in_features=1000, out_features=num_classes, bias=True)

        # Dropout layer
        self.drop = nn.Dropout(p=0.3)

        # Text processing (using the 768-dimensional BERT arrays)
        self.text_model = BertModel.from_pretrained("bert-base-uncased")

        # Text processing (Fully Connected Layer)
        self.fc_text = nn.Linear(in_features=self.text_model.config.hidden_size, out_features=num_classes, bias=True)

        # Fusion and classification
        self.softmax = nn.Softmax(dim=1)

    def forward(self, image, text_input_ids, text_attention_mask,):
        # Image branch
        x_img = self.image_model(image)
        x_img = self.drop(x_img)
        x_img = self.fc_image(x_img)

        # Text branch
        x_text_last_hidden_states = self.text_model(
            input_ids = text_input_ids,
            attention_mask = text_attention_mask,
            return_dict=False
        )
        x_text_pooled_output = x_text_last_hidden_states[0][:, 0, :]
        x_text = self.drop(x_text_pooled_output)
        x_text = self.fc_text(x_text_pooled_output)

        # Fusion and max merge
        x = torch.max(x_text, x_img)

        # Classification
        #x = self.softmax(x) #-> already applied in crossentropy loss

        return x

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BERTResNetClassifier(num_classes=6)
model= model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, min_lr=1e-6, factor=0.5, patience=1)
num_epochs = 20

train_model(model, train_loader,val_loader, criterion, optimizer, scheduler, num_epochs)
#print("\n")
evaluate_model(model, test_loader, criterion)

# Personal Notes on BERT


*   The decision to use an encoder-only architecture in BERT suggests a primary emphasis on understanding input sequences rather than generating output sequences.
*    BERT uses a bi-directional approach considering both the left and right context of words in a sentence, instead of analyzing the text sequentially
*   BERT is pre-trained on large amount of unlabeled text data. The model learns contextual embeddings, which are the representations of words that take into account their surrounding context in a sentence.
*   BERT is fine-tuned using labeled data specific to the downstream tasks of interest. These tasks could include sentiment analysis, question-answering, named entity recognition, or any other NLP application.


### Architecture:
*   BERT*BASE* has 12 layers in the Encoder stack while BERT*LARGE* has 24 layers in the Encoder stack.
*   BERT architectures (BASE and LARGE) also have larger feedforward networks (768 and 1024 hidden units respectively), and more attention heads (12 and 16 respectively)

---


*   https://www.geeksforgeeks.org/explanation-of-bert-model-nlp/
*   https://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/
*   https://jalammar.github.io/illustrated-bert/











# Personal Notes on ResNets

*   The groundbreaking contribution of ResNet is the introduction of the *residual block*. These residual blocks allow connecting the activations of previous layers with the next while skipping some layers in between, allowing the gradient to flow without being altered by a large magnitude.
*   ResNet blocks can result in dimensionality mismatches which we can fix by passing outputs through 1x1 convolutions with strides of 2 rather than adding extra padding
    - (ex: if we had 3 input channels and 6 output channels, we would pass input through 3 1x1 convolutional layers to double number of input channels)
    - (ex: stride of 2 ensures that 64x64 layer would become 32x32)


### ResNets vs VGG?
I also considered using VGG for Image Classification but saw upon research that ResNets were found to be more effective
*   ResNet50 is significantly deeper than VGG in terms of architecture. One of the drawbacks of VGG was that researchers couldn't go as deep as wanted, because they started to lose generalization capability.
*   ResNet50 had smoother training and faster convergence, thanks to the skip connections

# Early Fusion vs Late Fusion

- Early fusion approach combines raw data from multiple sensors before any high-level processing or decision-making. It helps capture and process interactions between modalities at the data level. 
    - The advantage here is that we don’t have to perform dedicated processing for each modality (i.e, it only requires a single learning phase)
    - The downside to this approach is that raw input data may not contain rich semantic information. This means that the model is not able to capture complex interactions between the modalities.
- Late fusion approach processes the data of each sensor independently to make a local prediction. These individual results are then combined at a higher level to make the final fused prediction.
    - The advantage of late fusion is its simplicity and isolation. Each model gets to learn super rich information on its modality.
    - The downside is that system is not able to learn complex modal interactions, and thus does not benefit directly from the complementary information each modality might offer.
    - Another downside is the high compute cost for processing data of each mode separately.

    
Ultimately I chose to do Late Fusion as it seemed to have generally better results for image-text models.

---
- https://medium.com/@raj.pulapakura/multimodal-models-and-fusion-a-complete-guide-225ca91f6861
- https://chaozhangchn.medium.com/performance-comparison-between-early-fusion-and-late-fusion-5f9d88ffce66
- https://dl.acm.org/doi/pdf/10.1145/3589335.3652504